In [ ]:
import pandas as pd
import numpy as np
from scipy.interpolate import griddata
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.cm import ScalarMappable
import contextily as ctx

In [ ]:
# ✅ Solução:
# Selecionar apenas datas comuns entre ASC e DESC.

# Para cada data comum:
# Interpolar espacialmente os dados da órbita ASC para os pontos da órbita DESC (ou vice-versa).
# Depois combinar os dois conjuntos em cada ponto da malha.
# Finalmente, calcular a média dos componentes up, east, north.

# 🧠 Estratégia
# Interpolar usando scipy.interpolate.griddata() só nas datas que existem nas duas órbitas.
# Sem interpolação temporal.
# Usar malha não regular: os pontos reais da outra órbita.

# Pontos = posições da órbita ascendente (mesmos easting e northing de df_asc);

# ========== 1. LER E FILTRAR DADOS ==========
def ler_e_filtrar(caminho_csv):
    df = pd.read_csv(caminho_csv)

    # Filtro espacial (Alqueva)
    norte_min, norte_max = 1855050, 1855850
    este_min, este_max = 2792250, 2793250

    df = df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) & (df['easting'] <= este_max)
    ]

    return df

df_asc = ler_e_filtrar("data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv")
df_desc = ler_e_filtrar("data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv")


### Interpolador linear:
- O método "linear" aqui cria uma triangulação e interpola linearmente, o que é bom, mas exige que os pontos estejam bem distribuídos. Pode falhar ou dar NaN se o ponto estiver fora do fecho convexo dos dados.

In [ ]:
# === 2. Melt componentes ===
def melt(df, comp):
    static = ['pid', 'easting', 'northing']
    dates = [c for c in df.columns if c.startswith('20')]
    df_temp = df[static + dates].copy()
    df_melt = df_temp.melt(id_vars=static, var_name='date', value_name=comp)
    df_melt['date'] = pd.to_datetime(df_melt['date'], format='%Y%m%d')
    return df_melt

def get_componentes(df, prefix):
    up = melt(df, f'up_{prefix}')
    east = melt(df, f'east_{prefix}')
    north = melt(df, f'north_{prefix}')
    merged = up.merge(east, on=['pid', 'easting', 'northing', 'date'])
    merged = merged.merge(north, on=['pid', 'easting', 'northing', 'date'])
    return merged

asc = get_componentes(df_asc, 'asc')
desc = get_componentes(df_desc, 'desc')

# === 3. Datas comuns ===
datas_comuns = np.intersect1d(asc['date'].unique(), desc['date'].unique())

# === 4. Interpolação espacial por data ===
resultados = []

for data in datas_comuns:
    asc_data = asc[asc['date'] == data]
    desc_data = desc[desc['date'] == data]

    # Interpolar ASC nos pontos DESC (LINEAR)
    pontos_asc = asc_data[['easting', 'northing']].values
    pontos_desc = desc_data[['easting', 'northing']].values

    interp = {}
    for comp in ['up_asc', 'east_asc', 'north_asc']:
        interp[comp] = griddata(
            pontos_asc,
            asc_data[comp].values,
            pontos_desc,
            method='linear'
        )

    # Criar DataFrame combinado
    df_comb = pd.DataFrame({
        'easting': desc_data['easting'].values,
        'northing': desc_data['northing'].values,
        'date': data,
        'up': np.nanmean([interp['up_asc'], desc_data['up_desc'].values], axis=0),
        'east': np.nanmean([interp['east_asc'], desc_data['east_desc'].values], axis=0),
        'north': np.nanmean([interp['north_asc'], desc_data['north_desc'].values], axis=0),
    })

    resultados.append(df_comb)

# === 5. Concatenar e salvar ===
df_final = pd.concat(resultados, ignore_index=True)
df_final.to_csv("alqueva_comb_espacial_linear.csv", index=False)


In [ ]:
# === 1. Carregar dados combinados ===
df = pd.read_csv("alqueva_comb_espacial_linear.csv")

# Calcular média por ponto
df_media = df.groupby(['easting', 'northing'], as_index=False).mean(numeric_only=True)

# === 2. Criar GeoDataFrame ===
gdf = gpd.GeoDataFrame(
    df_media,
    geometry=gpd.points_from_xy(df_media['easting'], df_media['northing']),
    crs="EPSG:3035"
)

# Converter para Web Mercator
gdf_web = gdf.to_crs(epsg=3857)

# === 3. Plot ===
componente = "up"  # ou 'east' ou 'north'
coluna_valor = componente
#cmap = plt.colormaps.get_cmap('jet')
cmap = plt.colormaps.get_cmap('jet').reversed()

# Escala fixa entre -20 e 20 mm
norma = colors.TwoSlopeNorm(
    vmin=-5,
    vcenter=0,
    vmax=5
)

fig, ax = plt.subplots(figsize=(10, 6))
gdf_web.plot(
    ax=ax,
    column=coluna_valor,
    cmap=cmap,
    markersize=10,
    norm=norma,
    legend=False
)

# Adiciona mapa base
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)

# Estilo
ax.set_axis_off()
plt.title(f"Deslocamento médio ({componente}) - ASC + DESC", fontsize=14)

# Barra de cores
sm = ScalarMappable(cmap=cmap, norm=norma)
sm._A = []
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Deslocamento (mm)", fontsize=12)

plt.tight_layout()
plt.show()


### Interpolador IDW:
- IDW pode ser computacionalmente mais pesado do que griddata, especialmente com muitos pontos.
- É sensível à escolha da potência (power). Normalmente entre 1.5 e 3 funciona bem.
- Não extrapola mal como o linear — mas pode alisar demasiado os dados se os pontos estiverem espaçados.

Abaixo tens uma versão do teu código com IDW no lugar de griddata, mantendo toda a lógica original, mas trocando apenas o método de interpolação espacial (ponto 4). Tudo o resto é exatamente igual.

O ficheiro final será salvo como: alqueva_comb_espacial_idw.csv

O parâmetro power=2 na função idw_interpolation() é ajustável. Quanto maior, mais peso é dado a pontos próximos.

Se quiseres comparar com o método linear, agora tens os dois scripts — basta mudar o nome do ficheiro no final e sobrepor os resultados.

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.cm import ScalarMappable
import contextily as ctx

# ========== 1. LER E FILTRAR DADOS ==========
def ler_e_filtrar(caminho_csv):
    df = pd.read_csv(caminho_csv)

    # Filtro espacial (Alqueva)
    norte_min, norte_max = 1855050, 1855850
    este_min, este_max = 2792250, 2793250

    df = df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) & (df['easting'] <= este_max)
    ]

    return df

df_asc = ler_e_filtrar("data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv")
df_desc = ler_e_filtrar("data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv")

# ========== 2. DERRETER COMPONENTES ==========
def melt(df, comp):
    static = ['pid', 'easting', 'northing']
    dates = [c for c in df.columns if c.startswith('20')]
    df_temp = df[static + dates].copy()
    df_melt = df_temp.melt(id_vars=static, var_name='date', value_name=comp)
    df_melt['date'] = pd.to_datetime(df_melt['date'], format='%Y%m%d')
    return df_melt

def get_componentes(df, prefix):
    up = melt(df, f'up_{prefix}')
    east = melt(df, f'east_{prefix}')
    north = melt(df, f'north_{prefix}')
    merged = up.merge(east, on=['pid', 'easting', 'northing', 'date'])
    merged = merged.merge(north, on=['pid', 'easting', 'northing', 'date'])
    return merged

asc = get_componentes(df_asc, 'asc')
desc = get_componentes(df_desc, 'desc')

# ========== 3. DATAS COMUNS ==========
datas_comuns = np.intersect1d(asc['date'].unique(), desc['date'].unique())

# ========== 4. INTERPOLAÇÃO ESPACIAL (IDW) ==========
def idw_interpolation(xy_known, values_known, xy_target, power=2):
    interpolated = []
    for x0, y0 in xy_target:
        dists = np.sqrt((xy_known[:, 0] - x0)**2 + (xy_known[:, 1] - y0)**2)

        if np.any(dists == 0):
            interpolated.append(values_known[dists == 0][0])
        else:
            weights = 1 / (dists ** power)
            weighted_sum = np.sum(weights * values_known)
            interpolated.append(weighted_sum / np.sum(weights))

    return np.array(interpolated)

resultados = []

for data in datas_comuns:
    asc_data = asc[asc['date'] == data]
    desc_data = desc[desc['date'] == data]

    pontos_asc = asc_data[['easting', 'northing']].values
    pontos_desc = desc_data[['easting', 'northing']].values

    interp = {}
    for comp in ['up_asc', 'east_asc', 'north_asc']:
        interp[comp] = idw_interpolation(
            pontos_asc,
            asc_data[comp].values,
            pontos_desc,
            power=2  # <-- Podes ajustar este valor
        )

    df_comb = pd.DataFrame({
        'easting': desc_data['easting'].values,
        'northing': desc_data['northing'].values,
        'date': data,
        'up': np.nanmean([interp['up_asc'], desc_data['up_desc'].values], axis=0),
        'east': np.nanmean([interp['east_asc'], desc_data['east_desc'].values], axis=0),
        'north': np.nanmean([interp['north_asc'], desc_data['north_desc'].values], axis=0),
    })

    resultados.append(df_comb)

# ========== 5. SALVAR COMBINADO ==========
df_final = pd.concat(resultados, ignore_index=True)
df_final.to_csv("alqueva_comb_espacial_idw.csv", index=False)

# ========== 6. CARREGAR E PLOTAR ==========
df = pd.read_csv("alqueva_comb_espacial_idw.csv")
df_media = df.groupby(['easting', 'northing'], as_index=False).mean(numeric_only=True)

gdf = gpd.GeoDataFrame(
    df_media,
    geometry=gpd.points_from_xy(df_media['easting'], df_media['northing']),
    crs="EPSG:3035"
)

gdf_web = gdf.to_crs(epsg=3857)

componente = "up"  # ou 'east' ou 'north'
coluna_valor = componente
cmap = plt.colormaps.get_cmap('jet').reversed()

norma = colors.TwoSlopeNorm(
    vmin=-5,
    vcenter=0,
    vmax=5
)

fig, ax = plt.subplots(figsize=(10, 6))
gdf_web.plot(
    ax=ax,
    column=coluna_valor,
    cmap=cmap,
    markersize=10,
    norm=norma,
    legend=False
)

ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)

ax.set_axis_off()
plt.title(f"Deslocamento médio ({componente}) - ASC + DESC (IDW)", fontsize=14)

sm = ScalarMappable(cmap=cmap, norm=norma)
sm._A = []
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Deslocamento (mm)", fontsize=12)

plt.tight_layout()
plt.show()


### Comparação interpolador linear VS IDW

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib.cm import ScalarMappable
import contextily as ctx

# === 1. Carregar os dois resultados ===
df_linear = pd.read_csv("alqueva_comb_espacial_linear.csv")
df_idw = pd.read_csv("alqueva_comb_espacial_idw.csv")

# === 2. Calcular médias por ponto ===
df_linear_media = df_linear.groupby(['easting', 'northing'], as_index=False).mean(numeric_only=True)
df_idw_media = df_idw.groupby(['easting', 'northing'], as_index=False).mean(numeric_only=True)

# === 3. Criar GeoDataFrames ===
gdf_linear = gpd.GeoDataFrame(
    df_linear_media,
    geometry=gpd.points_from_xy(df_linear_media['easting'], df_linear_media['northing']),
    crs="EPSG:3035"
)

gdf_idw = gpd.GeoDataFrame(
    df_idw_media,
    geometry=gpd.points_from_xy(df_idw_media['easting'], df_idw_media['northing']),
    crs="EPSG:3035"
)

# Converter para Web Mercator
gdf_linear = gdf_linear.to_crs(epsg=3857)
gdf_idw = gdf_idw.to_crs(epsg=3857)

# === 4. Parâmetros de plot ===
componente = "up"  # ou 'east' ou 'north'
vmin, vmax = -5, 5
cmap = plt.colormaps.get_cmap("jet").reversed()
norma = colors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

# === 5. Figura lado a lado com legenda horizontal ===
fig, axs = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

for ax, gdf_data, title in zip(
    axs,
    [gdf_linear, gdf_idw],
    ["Interpolação Linear", "Interpolação IDW"]
):
    gdf_data.plot(
        ax=ax,
        column=componente,
        cmap=cmap,
        markersize=10,
        norm=norma,
        legend=False
    )
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
    ax.set_title(title, fontsize=13)
    ax.set_axis_off()

sm = ScalarMappable(cmap=cmap, norm=norma)
sm._A = []
cbar = plt.colorbar(sm, ax=axs, orientation='horizontal', fraction=0.05, pad=0.05)
cbar.set_label(f"Deslocamento {componente.upper()} (mm)", fontsize=12)

plt.show()



### Comparação com interpolador linear com o ortho

In [ ]:
# ========== 1. LER E FILTRAR DADOS ==========
def ler_e_filtrar(caminho_csv):
    df = pd.read_csv(caminho_csv)

    # Filtro espacial (Alqueva)
    norte_min, norte_max = 1855050, 1855850
    este_min, este_max = 2792250, 2793250

    df = df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) & (df['easting'] <= este_max)
    ]

    return df

df_asc = ler_e_filtrar("data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv")
df_desc = ler_e_filtrar("data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv")

df_ortho_e = ler_e_filtrar("data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv")
df_ortho_u = ler_e_filtrar("data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv")

In [ ]:
# ========== IMPORTS NECESSÁRIOS ==========
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import contextily as ctx
from matplotlib import colors
from matplotlib.cm import ScalarMappable
from scipy.interpolate import griddata



import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import contextily as ctx
from matplotlib import colors
from matplotlib.cm import ScalarMappable

# === 0. Parâmetros configuráveis ===
vmin, vmax = -5, 5  # Limites da escala de deslocamento (mm)
componente = 'up'  # ou 'east'

# === 1. Função para filtrar ===
def ler_e_filtrar(caminho_csv):
    df = pd.read_csv(caminho_csv)
    norte_min, norte_max = 1855050, 1855850
    este_min, este_max = 2792250, 2793250
    df = df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) & (df['easting'] <= este_max)
    ]
    return df

# === 2. Carregar dados ORTHO filtrados ===
df_ortho_e = ler_e_filtrar("data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv")
df_ortho_u = ler_e_filtrar("data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv")

# === 3. Função para calcular média e criar GeoDataFrame ORTHO ===
def media_ortho(df_ortho, componente_nome):
    datas = [c for c in df_ortho.columns if c.startswith('20')]
    df_ortho['media'] = df_ortho[datas].mean(axis=1, skipna=True)
    gdf = gpd.GeoDataFrame(
        df_ortho[['easting', 'northing', 'media']],
        geometry=gpd.points_from_xy(df_ortho['easting'], df_ortho['northing']),
        crs="EPSG:3035"
    )
    gdf = gdf.rename(columns={'media': componente_nome})
    return gdf

# === 4. Criar gdf_ortho_web com o componente desejado ===
dict_df_ortho = {'up': df_ortho_u, 'east': df_ortho_e}
gdf_ortho = media_ortho(dict_df_ortho[componente], componente)
gdf_ortho_web = gdf_ortho.to_crs(epsg=3857)

# === 5. Carregar dados da fusão ASC+DESC já processados ===
df_fusao = pd.read_csv("alqueva_comb_espacial.csv")
df_media = df_fusao.groupby(['easting', 'northing'], as_index=False).mean(numeric_only=True)
gdf_fusao = gpd.GeoDataFrame(
    df_media,
    geometry=gpd.points_from_xy(df_media['easting'], df_media['northing']),
    crs="EPSG:3035"
)
gdf_web = gdf_fusao.to_crs(epsg=3857)

# === 6. Plot comparativo ===
cmap = plt.colormaps.get_cmap('jet').reversed()
norma = colors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

fig, axs = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

for ax, gdf_data, title in zip(
    axs,
    [gdf_web, gdf_ortho_web],
    ["Fusão ASC+DESC", "EGMS ORTHO"]
):
    gdf_data.plot(
        ax=ax,
        column=componente,
        cmap=cmap,
        markersize=10,
        norm=norma,
        legend=False
    )
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
    ax.set_title(title, fontsize=13)
    ax.set_axis_off()

sm = ScalarMappable(cmap=cmap, norm=norma)
sm._A = []
cbar = plt.colorbar(sm, ax=axs, orientation='horizontal', fraction=0.05, pad=0.05)
cbar.set_label(f"Deslocamento {componente.upper()} (mm)", fontsize=12)

plt.show()


### Comparação com interpolador IDW com o ortho

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from matplotlib import colors
from matplotlib.cm import ScalarMappable

# === 0. Parâmetros configuráveis ===
vmin, vmax = -5, 5  # Limites da escala de deslocamento (mm)
componente = 'up'  # ou 'east'

# === 1. Função para filtrar ===
def ler_e_filtrar(caminho_csv):
    df = pd.read_csv(caminho_csv)
    norte_min, norte_max = 1855050, 1855850
    este_min, este_max = 2792250, 2793250
    df = df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) & (df['easting'] <= este_max)
    ]
    return df

# === 2. Carregar dados ORTHO filtrados ===
df_ortho_e = ler_e_filtrar("data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv")
df_ortho_u = ler_e_filtrar("data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv")

# === 3. Função para calcular média e criar GeoDataFrame ORTHO ===
def media_ortho(df_ortho, componente_nome):
    datas = [c for c in df_ortho.columns if c.startswith('20')]
    df_ortho['media'] = df_ortho[datas].mean(axis=1, skipna=True)
    gdf = gpd.GeoDataFrame(
        df_ortho[['easting', 'northing', 'media']],
        geometry=gpd.points_from_xy(df_ortho['easting'], df_ortho['northing']),
        crs="EPSG:3035"
    )
    gdf = gdf.rename(columns={'media': componente_nome})
    return gdf

# === 4. Criar gdf_ortho_web com o componente desejado ===
dict_df_ortho = {'up': df_ortho_u, 'east': df_ortho_e}
gdf_ortho = media_ortho(dict_df_ortho[componente], componente)
gdf_ortho_web = gdf_ortho.to_crs(epsg=3857)

# === 5. Carregar dados IDW já processados ===
df_idw = pd.read_csv("alqueva_comb_espacial_idw.csv")
df_idw_media = df_idw.groupby(['easting', 'northing'], as_index=False).mean(numeric_only=True)
gdf_idw = gpd.GeoDataFrame(
    df_idw_media,
    geometry=gpd.points_from_xy(df_idw_media['easting'], df_idw_media['northing']),
    crs="EPSG:3035"
)
gdf_idw_web = gdf_idw.to_crs(epsg=3857)

# === 6. Plot comparativo ===
cmap = plt.colormaps.get_cmap('jet').reversed()
norma = colors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

fig, axs = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

for ax, gdf_data, title in zip(
    axs,
    [gdf_idw_web, gdf_ortho_web],
    ["Interpolação IDW", "EGMS ORTHO"]
):
    gdf_data.plot(
        ax=ax,
        column=componente,
        cmap=cmap,
        markersize=10,
        norm=norma,
        legend=False
    )
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
    ax.set_title(title, fontsize=13)
    ax.set_axis_off()

sm = ScalarMappable(cmap=cmap, norm=norma)
sm._A = []
cbar = plt.colorbar(sm, ax=axs, orientation='horizontal', fraction=0.07, pad=0.05)
cbar.set_label(f"Deslocamento {componente.upper()} (mm)", fontsize=12)

plt.suptitle("Comparação entre Interpolação IDW e ORTHO", fontsize=16)
plt.show()


In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from sklearn.metrics import mean_absolute_error

# === Parâmetro configurável ===
componente = 'up'  # ou 'east'

# === 1. Garante que os dois GeoDataFrames estão no mesmo CRS ===
gdf_web = gdf_web.to_crs(epsg=3857)
gdf_ortho_web = gdf_ortho_web.to_crs(epsg=3857)

# === 2. Renomeia coluna do ortho para evitar confusão ===
gdf_ortho_web = gdf_ortho_web.rename(columns={componente: f"{componente}_ortho"})

# === 3. Spatial Join (pontos mais próximos dentro de 50 metros) ===
gdf_joined = gpd.sjoin_nearest(
    gdf_web,
    gdf_ortho_web,
    how='inner',
    max_distance=50,   # em metros
    distance_col='dist'
)

print(f"✅ Número de pontos após spatial join: {len(gdf_joined)}")

# === 4. Remove linhas com valores nulos nas colunas de comparação ===
gdf_joined = gdf_joined.dropna(subset=[componente, f"{componente}_ortho"])

# === 5. Extrai valores para comparação ===
val_fusao = gdf_joined[componente].values
val_ortho = gdf_joined[f"{componente}_ortho"].values

# === 6. Calcula métricas ===
correl = np.corrcoef(val_fusao, val_ortho)[0, 1]
diff_media = np.mean(val_fusao - val_ortho)
mae = mean_absolute_error(val_ortho, val_fusao)

# === 7. Mostra os resultados ===
print("\n📊 Comparação entre Fusão ASC+DESC e ORTHO:")
print(f"🔹 Componente: {componente.upper()}")
print(f"🔹 Correlação (Pearson R): {correl:.3f}")
print(f"🔹 Diferença média (fusão - ortho): {diff_media:.2f} mm")
print(f"🔹 Erro absoluto médio (MAE): {mae:.2f} mm")
print(f"🔹 Nº de pontos comparados: {len(val_fusao)}")


## 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# === Scatter Plot ===
plt.figure(figsize=(8, 8))
plt.scatter(val_ortho, val_fusao, s=20, alpha=0.6, edgecolor='k', label='Pontos')

# Linha de referência y = x
lims = [
    np.min([val_ortho.min(), val_fusao.min()]),
    np.max([val_ortho.max(), val_fusao.max()])
]
plt.plot(lims, lims, 'r--', label='y = x')

# Estilo
plt.xlabel(f'Deslocamento ORTHO ({componente.upper()}) [mm]')
plt.ylabel(f'Deslocamento Fusão ASC+DESC ({componente.upper()}) [mm]')
plt.title(f'Comparação de Deslocamentos - {componente.upper()}')
plt.legend()
plt.grid(True)
plt.axis('square')
plt.xlim(lims)
plt.ylim(lims)
plt.tight_layout()
plt.show()



# 1. Explorar Outliers — pontos com maiores discrepâncias

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx

df_joined = gdf_joined.copy()

# Extrai coordenadas x e y da geometria
df_joined['easting'] = df_joined.geometry.x
df_joined['northing'] = df_joined.geometry.y

# Calcula o erro absoluto
df_joined['error_abs'] = np.abs(df_joined['up'] - df_joined['up_ortho'])

threshold = 3
outliers = df_joined[df_joined['error_abs'] > threshold]

print(f"Número de outliers (erro > {threshold} mm): {len(outliers)}")
print(outliers[['easting', 'northing', 'up', 'up_ortho', 'error_abs']])

# Plotar outliers no mapa
ax = outliers.plot(
    kind='scatter',
    x='easting',
    y='northing',
    c='error_abs',
    cmap='Reds',
    colorbar=True,
    figsize=(8, 6),
    s=50,
    alpha=0.7,
    edgecolor='k'
)
ctx.add_basemap(ax, crs=df_joined.crs)
ax.set_title("Outliers: Maiores discrepâncias no deslocamento (mm)")
plt.show()



## 2. Testar vários max_distance e ver métricas

In [ ]:
from sklearn.metrics import mean_absolute_error
import pandas as pd

distances = [10, 25, 50, 100, 200]  # em metros
results = []

for dist in distances:
    joined = gpd.sjoin_nearest(
        gdf_web,
        gdf_ortho_web,
        how='inner',
        max_distance=dist,
        distance_col='dist'
    )
    if joined.empty:
        continue
    
    joined['error_abs'] = np.abs(joined['up'] - joined['up_ortho'])
    
    mae = mean_absolute_error(joined['up_ortho'], joined['up'])
    correl = np.corrcoef(joined['up'], joined['up_ortho'])[0, 1]
    
    results.append({
        'max_distance': dist,
        'n_points': len(joined),
        'mae': mae,
        'correlation': correl
    })

df_results = pd.DataFrame(results)
print(df_results)


### 3. Cálculo de RMSE e R²

In [ ]:
import numpy as np
from sklearn.metrics import r2_score

up_fused = df_joined['up'].values
up_ortho = df_joined['up_ortho'].values

rmse = np.sqrt(np.mean((up_ortho - up_fused)**2))
r2 = r2_score(up_ortho, up_fused)

print(f"RMSE: {rmse:.2f} mm")
print(f"R²: {r2:.3f}")

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.cm import ScalarMappable
from scipy.interpolate import griddata
import contextily as ctx

# ========== 1. LER E FILTRAR DADOS ==========

def ler_e_filtrar(caminho_csv):
    df = pd.read_csv(caminho_csv)

    # Filtro espacial (Alqueva)
    norte_min, norte_max = 1855050, 1855850
    este_min, este_max = 2792250, 2793250

    df = df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) & (df['easting'] <= este_max)
    ]

    return df

# Carregar dados ASC/DESC/ORTHO
df_asc = ler_e_filtrar("data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv")
df_desc = ler_e_filtrar("data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv")
df_ortho_u = ler_e_filtrar("data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv")

# ========== 2. Melt componentes ==========

def melt(df, comp):
    static = ['pid', 'easting', 'northing']
    dates = [c for c in df.columns if c.startswith('20')]
    df_temp = df[static + dates].copy()
    df_melt = df_temp.melt(id_vars=static, var_name='date', value_name=comp)
    df_melt['date'] = pd.to_datetime(df_melt['date'], format='%Y%m%d')
    return df_melt

def get_componentes(df, prefix):
    up = melt(df, f'up_{prefix}')
    east = melt(df, f'east_{prefix}')
    north = melt(df, f'north_{prefix}')
    merged = up.merge(east, on=['pid', 'easting', 'northing', 'date'])
    merged = merged.merge(north, on=['pid', 'easting', 'northing', 'date'])
    return merged

asc = get_componentes(df_asc, 'asc')
desc = get_componentes(df_desc, 'desc')

# ========== 3. Datas comuns ==========
datas_comuns = np.intersect1d(asc['date'].unique(), desc['date'].unique())

# ========== 4. Interpolação com fallback ==========

def interpolar_com_fallback(pontos_orig, valores, pontos_dest):
    resultado = griddata(pontos_orig, valores, pontos_dest, method='linear')
    if np.isnan(resultado).all():
        resultado = griddata(pontos_orig, valores, pontos_dest, method='nearest')
    return resultado

resultados = []

for data in datas_comuns:
    asc_data = asc[asc['date'] == data]
    desc_data = desc[desc['date'] == data]

    pontos_asc = asc_data[['easting', 'northing']].values
    pontos_desc = desc_data[['easting', 'northing']].values

    interp = {}
    for comp in ['up_asc', 'east_asc', 'north_asc']:
        interp[comp] = interpolar_com_fallback(
            pontos_asc,
            asc_data[comp].values,
            pontos_desc
        )

    df_comb = pd.DataFrame({
        'easting': desc_data['easting'].values,
        'northing': desc_data['northing'].values,
        'date': data,
        'up': np.nanmean([interp['up_asc'], desc_data['up_desc'].values], axis=0),
        'east': np.nanmean([interp['east_asc'], desc_data['east_desc'].values], axis=0),
        'north': np.nanmean([interp['north_asc'], desc_data['north_desc'].values], axis=0),
    })

    if df_comb['up'].isna().all():
        print(f"⚠️ Todos os valores 'up' são NaN na data {data.date()}")

    resultados.append(df_comb)

df_final = pd.concat(resultados, ignore_index=True)
df_final.to_csv("alqueva_comb_espacial.csv", index=False)

print("✅ Exportado: alqueva_comb_espacial.csv")

# ========== 5. Juntar com ORTHO e plotar ==========

# ASC+DESC: média por ponto
df_media = df_final.groupby(['easting', 'northing'], as_index=False).mean(numeric_only=True)

gdf_fusao = gpd.GeoDataFrame(
    df_media,
    geometry=gpd.points_from_xy(df_media['easting'], df_media['northing']),
    crs="EPSG:3035"
).to_crs(3857)

# ORTHO: média da coluna 'up'
df_ortho_u['up_ortho'] = df_ortho_u[[c for c in df_ortho_u.columns if c.startswith('20')]].mean(axis=1)
gdf_ortho = gpd.GeoDataFrame(
    df_ortho_u,
    geometry=gpd.points_from_xy(df_ortho_u['easting'], df_ortho_u['northing']),
    crs="EPSG:3035"
).to_crs(3857)

# Spatial Join por proximidade
gdf_joined = gpd.sjoin_nearest(
    gdf_fusao, gdf_ortho,
    how='inner',
    max_distance=50,
    distance_col='dist'
)

# ========== 6. Plot comparativo ==========

componente = "up"
cmap = plt.cm.jet.reversed()
norma = colors.TwoSlopeNorm(vmin=-5, vcenter=0, vmax=5)

fig, ax = plt.subplots(figsize=(10, 6))

gdf_joined.plot(
    ax=ax,
    column='up',
    cmap=cmap,
    markersize=20,
    alpha=0.7,
    norm=norma,
    label="ASC+DESC"
)

gdf_joined.plot(
    ax=ax,
    column='up_ortho',
    cmap=cmap,
    markersize=10,
    marker='x',
    alpha=0.9,
    norm=norma,
    label="ORTHO"
)

ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_axis_off()
plt.title("Deslocamento médio (UP) - Fusão ASC+DESC vs ORTHO", fontsize=14)
sm = ScalarMappable(cmap=cmap, norm=norma)
plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02).set_label("Deslocamento (mm)", fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()


### 1. Mapa de diferenças (heatmap)
- os métodos concordam (diferença próxima de zero) ou divergem (diferenças positivas ou negativas grandes)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors

# Calcular a diferença
gdf_joined['dif_up'] = gdf_joined['up'] - gdf_joined['up_ortho']

# Definir norma para o colormap centrado no zero
norma_dif = colors.TwoSlopeNorm(vmin=-5, vcenter=0, vmax=5)

fig, ax = plt.subplots(figsize=(10, 7))
gdf_joined.plot(
    column='dif_up',
    cmap='RdBu_r',
    markersize=30,
    alpha=0.8,
    norm=norma_dif,
    legend=True,
    ax=ax
)

import contextily as ctx
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_axis_off()
plt.title("Diferença no deslocamento UP (Fusão ASC+DESC - ORTHO)", fontsize=14)
plt.show()


### Gráfico de dispersão (scatter plot) com linha y = x

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

up_fused = gdf_joined['up'].values
up_ortho = gdf_joined['up_ortho'].values

plt.figure(figsize=(8, 8))
plt.scatter(up_ortho, up_fused, s=20, alpha=0.6, edgecolor='k', label='Pontos')

lims = [
    np.min([up_ortho.min(), up_fused.min()]),
    np.max([up_ortho.max(), up_fused.max()])
]

plt.plot(lims, lims, 'r--', label='y = x')
plt.xlabel('Deslocamento ORTHO (mm)')
plt.ylabel('Deslocamento Fusão ASC+DESC (mm)')
plt.title('Comparação de Deslocamentos')
plt.legend()
plt.grid(True)
plt.axis('square')
plt.xlim(lims)
plt.ylim(lims)
plt.show()


### Histograma da diferença

In [ ]:
import matplotlib.pyplot as plt

gdf_joined['dif_up'] = gdf_joined['up'] - gdf_joined['up_ortho']

plt.figure(figsize=(8, 5))
plt.hist(gdf_joined['dif_up'], bins=50, color='steelblue', edgecolor='black')
plt.axvline(0, color='red', linestyle='--')
plt.title('Histograma das Diferenças (Fusão - ORTHO)')
plt.xlabel('Diferença de deslocamento (mm)')
plt.ylabel('Frequência')
plt.show()


In [ ]:
gdf_web = gdf.to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(8, 6))

gdf_web[gdf_web["cluster"] != -1].plot(
    ax=ax,
    column="cluster",
    cmap="tab10",
    markersize=8,
    legend=True,
    categorical=True
)

ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)

ax.set_axis_off()
plt.title("Clusters de Séries Temporais - up_fusion", fontsize=14)
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from sklearn.cluster import KMeans
from matplotlib import cm
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec

# Número de clusters
n_clusters = 5

# --- Criar GeoDataFrame ---
gdf = gpd.GeoDataFrame(
    df_asc,
    geometry=gpd.points_from_xy(df_asc.easting, df_asc.northing),
    crs="EPSG:3035"
)
gdf_web = gdf.to_crs(epsg=3857)

# --- KMeans clustering com coordenadas ---
coords = np.vstack([gdf_web.geometry.x, gdf_web.geometry.y]).T
kmeans = KMeans(n_clusters=n_clusters, random_state=0).fit(coords)
gdf_web["cluster"] = kmeans.labels_

# --- Preparar tempo (colunas do up_fusion) ---
tempo = pd.to_datetime(up_fusion.columns.astype(str), format="%Y%m%d")

# --- Séries temporais do up_fusion ---
serie_temporal = up_fusion.values  # shape (n_pontos, n_timestamps)

# --- Cores para clusters ---
cmap = cm.get_cmap('tab10', n_clusters)
cores_clusters = [cmap(i) for i in range(n_clusters)]

# --- Layout matplotlib gridspec ---
ncols = 2
nrows = int(np.ceil(n_clusters / ncols))

fig = plt.figure(figsize=(20, 5 + 3 * nrows))
gs = gridspec.GridSpec(nrows + 1, ncols, height_ratios=[2] + [1]*nrows, hspace=0.4, wspace=0.3)

# --- Mapa na primeira linha (ocupando as 2 colunas) ---
ax_mapa = fig.add_subplot(gs[0, :])
gdf_web.plot(ax=ax_mapa, column="cluster", categorical=True, legend=True, cmap="tab10", markersize=10)
ctx.add_basemap(ax_mapa, source=ctx.providers.Esri.WorldImagery)
ax_mapa.set_axis_off()
ax_mapa.set_title(f"Clusters espaciais (KMeans, {n_clusters} clusters)", fontsize=18)

# --- Plots para cada cluster ---
clusters_validos = np.unique(gdf_web["cluster"])

for i, cluster_id in enumerate(clusters_validos):
    row = (i // ncols) + 1  # +1 para considerar a linha do mapa
    col = i % ncols
    ax = fig.add_subplot(gs[row, col])
    
    # Índices dos pontos do cluster
    idx = gdf_web["cluster"] == cluster_id
    
    # Séries temporais correspondentes a esses pontos
    series_cluster = serie_temporal[idx.values, :]
    
    # Plotar séries individuais (cinza claro)
    for serie in series_cluster:
        ax.plot(tempo, serie, color='lightgray', linewidth=0.5, alpha=0.5)
    
    # Plotar a média do cluster (linha colorida)
    media_cluster = np.nanmean(series_cluster, axis=0)
    ax.plot(tempo, media_cluster, color=cores_clusters[cluster_id], linewidth=3, label=f"Média Cluster {cluster_id}")
    
    ax.set_title(f"Cluster {cluster_id} ({series_cluster.shape[0]} séries)", fontsize=12)
    ax.grid(True)
    ax.legend()
    ax.set_ylabel("Deslocamento (m)")
    
    # Ajustar formato do eixo x para datas
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=8))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

# Colocar label no eixo X somente na última linha
for ax in fig.get_axes()[-ncols:]:
    ax.set_xlabel("Tempo", fontsize=12)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from sklearn.cluster import KMeans

# === CONFIGURAÇÃO ===
componente = up_fusion
nome_componente = "up_fusion"

# === MÉDIA DO DESLOCAMENTO ===
df_asc[f"mean_{nome_componente}"] = componente.mean(axis=1)

# === CRIAÇÃO DO GeoDataFrame ===
gdf = gpd.GeoDataFrame(
    df_asc,
    geometry=gpd.points_from_xy(df_asc.easting, df_asc.northing),
    crs="EPSG:3035"
)
gdf_web = gdf.to_crs(epsg=3857)

# === PREPARAR DADOS PARA KMEANS ===
coluna_valor = f"mean_{nome_componente}"
gdf_kmeans = gdf_web.dropna(subset=[coluna_valor]).copy()

X = gdf_kmeans[[coluna_valor]].values  # Usar apenas o deslocamento médio para clustering

# === APLICAR KMEANS ===
n_clusters = 4  # Ajusta conforme desejado
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
gdf_kmeans["cluster"] = kmeans.fit_predict(X)

# === VISUALIZAÇÃO ===
fig, ax = plt.subplots(figsize=(12, 10))

gdf_kmeans.plot(
    ax=ax,
    column="cluster",
    categorical=True,
    cmap="Set1",
    markersize=10,
    legend=True
)

ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_axis_off()
ax.set_title(f"KMeans baseado no deslocamento médio ({coluna_valor})", fontsize=14)

plt.show()


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.cm import ScalarMappable
from sklearn.cluster import KMeans
import contextily as ctx

# === CONFIGURAÇÃO ===
componente = up_fusion
nome_componente = "up_fusion"

# === MÉDIA DO DESLOCAMENTO ===
df_asc[f"mean_{nome_componente}"] = componente.mean(axis=1)

# === CRIAÇÃO DO GeoDataFrame ===
gdf = gpd.GeoDataFrame(
    df_asc,
    geometry=gpd.points_from_xy(df_asc.easting, df_asc.northing),
    crs="EPSG:3035"
)
gdf_web = gdf.to_crs(epsg=3857)

# === PREPARAR DADOS PARA KMEANS ===
coluna_valor = f"mean_{nome_componente}"
gdf_kmeans = gdf_web.dropna(subset=[coluna_valor]).copy()

X = gdf_kmeans[[coluna_valor]].values  # Apenas deslocamento

# === APLICAR KMEANS ===
n_clusters = 4  # Ajuste conforme necessário
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
gdf_kmeans["cluster"] = kmeans.fit_predict(X)

# === VISUALIZAÇÃO ===
fig, ax = plt.subplots(figsize=(12, 10))

# Usar colormap típico do InSAR (jet)
cmap = plt.cm.jet
norm = colors.Normalize(vmin=gdf_kmeans[coluna_valor].min(), vmax=gdf_kmeans[coluna_valor].max())

# Plotar os clusters
gdf_kmeans.plot(
    ax=ax,
    column="cluster",
    cmap=cmap,
    markersize=10,
    legend=False  # Legenda feita manualmente abaixo
)

ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_axis_off()
ax.set_title(f"KMeans baseado apenas no deslocamento médio ({coluna_valor})", fontsize=14)

# === Criar legenda personalizada para cada cluster ===
handles = []
for cluster_id in range(n_clusters):
    cluster_data = gdf_kmeans[gdf_kmeans["cluster"] == cluster_id]
    cluster_min = cluster_data[coluna_valor].min()
    cluster_max = cluster_data[coluna_valor].max()
    handles.append(plt.Line2D(
        [0], [0], marker='o', color='w',
        label=f"Cluster {cluster_id}: {cluster_min:.2f} a {cluster_max:.2f} m",
        markerfacecolor=cmap(cluster_id / n_clusters), markersize=10
    ))

ax.legend(handles=handles, loc='upper left', title="Clusters e Limites", fontsize=10)

# Adicionar barra de cores para o deslocamento
sm = ScalarMappable(cmap=cmap, norm=norm)
sm._A = []  # Necessário para ScalarMappable funcionar
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label(f"{coluna_valor} (m)", fontsize=12)

plt.show()


In [ ]:
df_nivel = pd.read_excel("data/nivel.xlsx")
df_nivel